# 00 — Setup (AWS SageMaker)

**OPG-Live** — versi SageMaker dari `00_setup.ipynb` (yang lama Colab-only).

Instance: **ml.g5.xlarge** (A10G 24GB). Edit kode di VS Code → push GitHub → **Run All di sini**.

Beda vs Colab:
- Tidak ada `drive.mount()` — data hidup di **EBS** `~/SageMaker/` (persist saat Stop/Start).
- Backup durable = **S3** (pengganti Google Drive).
- Secret LLM dibaca dari file `~/SageMaker/.orkey`, bukan `google.colab.userdata`.

Urutan: paths → deps → SAM ViT-H → **upload DENTEX** → verify → LLM key → S3 backup.

## Cell 1 — Paths (EBS) + clone/pull repo + cek GPU

`DATA_ROOT` = pengganti `DRIVE_ROOT` lama. Semua data + checkpoint di sini, di EBS yang persisten.

In [ ]:
import os, subprocess, torch

# === EBS persistent root (survive Stop/Start; hilang hanya kalau instance di-DELETE) ===
DATA_ROOT = os.path.expanduser('~/SageMaker/opg-data')      # data + checkpoints + outputs
REPO      = os.path.expanduser('~/SageMaker/opg-live')      # kode

for sub in ['checkpoints', 'data/dentex', 'data/kb', 'outputs/masks', 'outputs/reports', 'outputs/metrics']:
    os.makedirs(f'{DATA_ROOT}/{sub}', exist_ok=True)

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/AndreasTopuh/opg-live.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)
%cd {REPO}/scripts

print('DATA_ROOT :', DATA_ROOT)
print('CUDA      :', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU       :', torch.cuda.get_device_name(0), f'| {total/1e9:.1f} GB total, {free/1e9:.1f} GB free')
    assert total/1e9 > 6, 'VRAM < 6GB — butuh GPU lebih besar untuk SAM ViT-H'

## Cell 2 — Install dependencies (~3-5 menit)

In [ ]:
%pip install -q segment-anything pycocotools pdfplumber \
    sentence-transformers openai numpy opencv-python-headless scipy \
    scikit-learn statsmodels matplotlib gdown
print('Core deps installed.')

## Cell 3 — Download SAM ViT-H (~2.5 GB, sekali; cache ke EBS)

In [ ]:
import os, urllib.request
CKPT = f'{DATA_ROOT}/checkpoints/sam_vit_h_4b8939.pth'
URL  = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
if not os.path.exists(CKPT):
    print('Downloading SAM ViT-H ke EBS...')
    urllib.request.urlretrieve(URL, CKPT)
print('SAM checkpoint:', CKPT, f'({os.path.getsize(CKPT)/1e9:.2f} GB)')

## Cell 4 — Upload DENTEX (PILIH SATU cara)

**Tujuan akhir:** folder `quadrant-enumeration-disease/` (JSON `*disease*.json` + `xrays/`) harus mendarat di
`~/SageMaker/opg-data/data/dentex/`. **Cuma split disease ini yang dibutuhkan Stage 2** (705 gambar, 2.7GB) — quadrant/enumeration/unlabelled TIDAK perlu.

**Opsi A (paling gampang — reuse zip yang sudah di Google Drive, pakai `gdown`):**
1. Di Google Drive, klik-kanan zip DENTEX → Share → *Anyone with the link*.
2. Ambil FILE_ID dari link `https://drive.google.com/file/d/FILE_ID/view`.
3. Isi `GDRIVE_ID` di sel bawah, jalankan.

**Opsi B (via S3 — cara AWS yang durable):** upload zip ke S3 (console drag-drop robust untuk file besar), lalu:
`!aws s3 cp s3://opg-live-235665523776/dentex_disease.zip {DATA_ROOT}/data/dentex/`

**Opsi C (drag-drop Jupyter):** hanya kalau sudah di-zip & kamu sabar (2.7GB rawan putus di browser).

In [ ]:
import os, glob
DENTEX_DIR = f'{DATA_ROOT}/data/dentex'

# --- Opsi A: gdown dari Google Drive (isi ID lalu un-comment) ---
# GDRIVE_ID = 'GANTI_DENGAN_FILE_ID'
# !cd {DENTEX_DIR} && gdown {GDRIVE_ID} -O dentex_disease.zip

# --- Opsi B: pull dari S3 (un-comment kalau zip sudah di S3) ---
# !aws s3 cp s3://opg-live-235665523776/dentex_disease.zip {DENTEX_DIR}/

# --- Unzip (jalankan setelah salah satu opsi di atas menaruh dentex_disease.zip) ---
zips = glob.glob(f'{DENTEX_DIR}/*.zip')
if zips:
    !cd {DENTEX_DIR} && unzip -o -q {os.path.basename(zips[0])} && echo 'unzipped'
else:
    print('Belum ada .zip di', DENTEX_DIR, '— jalankan Opsi A / B / C dulu.')
print('Isi data/dentex:')
!ls -R {DENTEX_DIR} | head -30

## Cell 5 — Verify DENTEX (harus 3529 polygon, distribusi cocok)

In [ ]:
import glob, json
from collections import Counter

# DENTEX disease = HIERARKIS: category_id_1=kuadran, _2=nomor gigi, _3=PENYAKIT (ada di categories_3).
cands = glob.glob(f'{DATA_ROOT}/data/dentex/**/*disease*.json', recursive=True)
print('File disease:', cands)
if not cands:
    print('⚠️  Belum ada *disease*.json — selesaikan Cell 4 dulu.')
else:
    d = json.load(open(cands[0]))
    cats3 = {c['id']: c['name'] for c in d.get('categories_3', [])}
    seg = [a for a in d['annotations'] if a.get('segmentation')]
    dist = Counter(a['category_id_3'] for a in seg)
    print('File         :', cands[0].split('/')[-1])
    print('Categories_3 :', cats3)
    print('Polygon (seg):', len(seg))
    print('Distribution :', {cats3.get(k, k): v for k, v in dist.items()})
    assert len(seg) == 3529, f'Polygon != 3529 (dapat {len(seg)})'
    assert dist[1] == 2189 and dist[0] == 604 and dist[3] == 578 and dist[2] == 158, 'Distribusi tidak cocok'
    print('\n✅ DENTEX terverifikasi: 3529 polygon, distribusi cocok.')

## Cell 6 — Test load SAM ViT-H di GPU

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
sam = sam_model_registry['vit_h'](checkpoint=CKPT).to('cuda')
predictor = SamPredictor(sam)
n = sum(p.numel() for p in sam.parameters())
print(f'✅ SAM ViT-H loaded di GPU — {n/1e6:.0f}M params. Stage 2 siap.')
del sam, predictor; torch.cuda.empty_cache()

## Cell 7 — LLM key (GPT-4o via OpenRouter, Stage 3)

Ganti `google.colab.userdata`. Bikin file sekali di **Terminal**:
```bash
echo 'sk-or-v1-...' > ~/SageMaker/.orkey && chmod 600 ~/SageMaker/.orkey
```
File `.orkey` ada di EBS (persist) tapi JANGAN di-commit ke git.

In [ ]:
import os
from openai import OpenAI
keyfile = os.path.expanduser('~/SageMaker/.orkey')
try:
    os.environ['OPENROUTER_API_KEY'] = open(keyfile).read().strip()
    client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
    r = client.chat.completions.create(model='openai/gpt-4o',
        messages=[{'role': 'user', 'content': 'reply with: OK'}], max_tokens=5)
    print('✅ OpenRouter OK — GPT-4o reply:', r.choices[0].message.content)
except FileNotFoundError:
    print('⚠️  Bikin dulu', keyfile, "(lihat instruksi sel markdown di atas).")
except Exception as e:
    print('⚠️  Key ada tapi gagal:', e)

## Cell 8 — S3 backup (pengganti Google Drive)

Sekali: bikin bucket. Lalu sync checkpoint/outputs ke S3 — aman walau instance di-Delete.
Restore kapan saja: balik arah `aws s3 sync`.

In [ ]:
BUCKET = 's3://opg-live-235665523776'   # akun-mu; ganti kalau nama bentrok
!aws s3 mb {BUCKET} 2>/dev/null; echo 'bucket siap (atau sudah ada)'

def backup():
    !aws s3 sync {DATA_ROOT}/checkpoints {BUCKET}/checkpoints
    !aws s3 sync {DATA_ROOT}/outputs     {BUCKET}/outputs
    print('✅ Backed up ke', BUCKET)

def restore():
    !aws s3 sync {BUCKET}/checkpoints {DATA_ROOT}/checkpoints
    !aws s3 sync {BUCKET}/outputs     {DATA_ROOT}/outputs
    print('✅ Restored dari', BUCKET)

print('Panggil backup() setelah training, restore() di instance baru.')

## Cell 9 — Train Stage 2 (script sama, path EBS, A10G 24GB → bs 1)

Smoke test 2 epoch dulu; kalau loss turun → naikkan `--epochs`. Checkpoint terbaik otomatis ke `{DATA_ROOT}/checkpoints/adapter_best.pth`.

In [ ]:
# SMOKE TEST (2 epoch) — pastikan loss turun & val Dice masuk akal
!python stage2/train_adapter.py \
  --drive {DATA_ROOT} \
  --sam_ckpt {DATA_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 2 --bs 1 --accum 8 --lr 1e-4

In [ ]:
# FULL RUN (setelah smoke test OK). Kalau OOM, tetap --bs 1. Kalau VRAM lega, coba --bs 2 --accum 4.
!python stage2/train_adapter.py \
  --drive {DATA_ROOT} \
  --sam_ckpt {DATA_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 8 --bs 1 --accum 8 --lr 1e-4
backup()   # langsung amankan ke S3

## ✅ Checklist

- [ ] Cell 1: paths EBS + repo pulled + GPU (A10G 24GB) terdeteksi
- [ ] Cell 2: deps installed
- [ ] Cell 3: SAM ViT-H di `{DATA_ROOT}/checkpoints`
- [ ] Cell 4: DENTEX disease split di `data/dentex/` (705 img)
- [ ] Cell 5: 3529 polygon terverifikasi
- [ ] Cell 6: SAM load di GPU OK
- [ ] Cell 7: OpenRouter key OK
- [ ] Cell 8: bucket S3 siap
- [ ] Cell 9: smoke test loss turun → full run → `adapter_best.pth` + `backup()`

⚠️ **STOP instance** setelah selesai (credit kepakai selama notebook nyala).

**Next:** `03_stage1_yolo` → `05_stage1_enum` → `02_make_artifacts` → `04_explain_rag`.